In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
meshki = pd.read_csv("meshki_dataset.csv")
meshki.head()

In [ ]:
selected_column = ['commentsCount', 'likesCount', 'timestamp', 'type', 'videoPlayCount', 'videoViewCount', 'videoDuration']

In [ ]:
meshki = meshki[selected_column]
meshki.head()

In [ ]:
unique_categ = meshki['type'].value_counts()
unique_categ

Sidecar post is essentially just a carousel of multiple images and are thus labelled as Image combined with other single images moving forward.

In [ ]:
meshki['type'] = meshki['type'].replace({
    'Sidecar': 'image',
    'Image': 'image',
    'Video': 'video'

})

In [ ]:
meshki.head()

In [ ]:
meshki['videoEngagementRate'] = (meshki['likesCount'] + meshki['commentsCount']*2) / meshki['videoViewCount'] * 100
meshki.head()


In [ ]:
#checking if there are any NaNs in videoEngagementRate problem

nan_count = (
    meshki.loc[meshki['type'] == 'video', 'videoEngagementRate']
    .isna()
    .sum()
)

In [ ]:
meshki['weighted_engagement'] = (meshki['likesCount'] + meshki['commentsCount']*2)

In [ ]:
WINDOW = 25
image_mask = meshki['type'] == 'image'

def rolling_zscore_min(series, window, min_periods=2):
    mean = series.rolling(window = window, min_periods = min_periods).mean()
    std = series.rolling(window = window, min_periods = min_periods).std()
    return (series - mean) / std

meshki['weighted_engagement_rate'] = np.nan
meshki.loc[image_mask, 'weighted_engagement_rate'] = rolling_zscore_min(
    meshki.loc[image_mask, 'weighted_engagement'],
    window=WINDOW,
    min_periods=5
)

In [ ]:
# NaNs if weighted_engagement itself has missing values

meshki.loc[image_mask, 'weighted_engagement_rate'].isna().sum()


In [ ]:
dist_data = meshki.loc[
    (meshki['type'] == 'image') &
    (~meshki['weighted_engagement_rate'].isna()),
    'weighted_engagement_rate'
]

plt.figure()
plt.hist(dist_data, bins=30)
plt.xlabel('Weighted Engagement Rate (Rolling Z-score)')
plt.ylabel('Frequency')
plt.title('Distribution of Weighted Engagement Rate (Images, Window=20)')
plt.show()

In [ ]:
dist_data.describe()

In [ ]:
meshki = meshki[meshki['videoViewCount'] != 0]

(meshki['videoViewCount'] == '0.0').sum()

## Separating datasets for video and images

In [ ]:
meshki_video = meshki[meshki['type'] == 'video']
meshki_image = meshki[meshki['type'] == 'image']

## Running first model: Prophet

In [ ]:
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics


In [ ]:
meshki['timestamp'] = (
    pd.to_datetime(meshki['timestamp'], utc=True)
      .dt.tz_convert('Australia/Sydney')
)

In [ ]:
# adding holiday seasons

holidays = pd.DataFrame([
    # --- Black Friday ---
    {
        'holiday': 'black_friday',
        'ds': '2024-11-29',
        'lower_window': -5,
        'upper_window': 2
    },
    {
        'holiday': 'black_friday',
        'ds': '2025-11-28',
        'lower_window': -5,
        'upper_window': 2
    },

    # --- Christmas ---
    {
        'holiday': 'christmas',
        'ds': '2024-12-25',
        'lower_window': -10,
        'upper_window': 5
    },
    {
        'holiday': 'christmas',
        'ds': '2025-12-25',
        'lower_window': -10,
        'upper_window': 5
    },

    # --- Valentine’s Day ---
    {
        'holiday': 'valentines_day',
        'ds': '2025-02-14',
        'lower_window': -5,
        'upper_window': 1
    },

    # --- End of Summer (fashion seasonality, not a fixed holiday) ---
    {
        'holiday': 'end_of_summer',
        'ds': '2024-08-31',
        'lower_window': -14,
        'upper_window': 7
    },
    {
        'holiday': 'end_of_summer',
        'ds': '2025-08-31',
        'lower_window': -14,
        'upper_window': 7
    },

    # --- Winter Season kickoff ---
    {
        'holiday': 'winter_season',
        'ds': '2024-11-01',
        'lower_window': 0,
        'upper_window': 30
    },
    {
        'holiday': 'winter_season',
        'ds': '2025-11-01',
        'lower_window': 0,
        'upper_window': 30
    },
])

# Ensure datetime
holidays['ds'] = pd.to_datetime(holidays['ds'])

In [ ]:
video_df = meshki_video[['timestamp', 'videoEngagementRate']].dropna()

video_prophet = (
    meshki_video[['timestamp', 'videoEngagementRate']]
    .dropna()
    .rename(columns={'timestamp': 'ds', 'videoEngagementRate': 'y'})
)

# Strip timezone AFTER converting
video_prophet['ds'] = (
    pd.to_datetime(video_prophet['ds'])
      .dt.tz_localize(None)
)

m_video = Prophet(
    changepoint_prior_scale=0.06,  # Smaller values make trends smoother. Larger values detect more shifts.
    seasonality_prior_scale=10,    # Adjusts how strongly seasonal patterns are fitted
    holidays=holidays,
    seasonality_mode='multiplicative',
    weekly_seasonality=True,
    yearly_seasonality=True
)

m_video.fit(video_prophet)


In [ ]:
future_video = m_video.make_future_dataframe(periods=30, freq='D')
future_video['ds'] = future_video['ds'].dt.tz_localize(None)


forecast_video = m_video.predict(future_video)


In [ ]:
fig1 = m_video.plot(forecast_video)
plt.title('Video Engagement Rate Forecast')
plt.ylabel('Engagement Rate (%)')
plt.show()

In [ ]:
fig2 = m_video.plot_components(forecast_video)
plt.show()

In [ ]:
last_date = video_prophet['ds'].max()

forecast_only = forecast_video[forecast_video['ds'] > last_date]
plt.figure(figsize=(10, 5))
plt.plot(
    forecast_only['ds'],
    forecast_only['yhat'],
    label='Forecast',
    linewidth=2
)

## Cross‑validation (to evaluate Prophet, for later comparison)

In [ ]:
df_cv_video = cross_validation(
    m_video,
    initial='365 days',   # first training window
    period='30 days',     # step between cutoffs
    horizon='30 days'     # forecast horizon
)
video_perf = performance_metrics(df_cv_video)
video_perf[['horizon', 'mape', 'rmse', 'mae']].head()

## ARIMA/SARIMA

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
#unit root test to test stationarity

def adf_test(series, name='series'):
    """Print ADF test result to decide if differencing is needed."""
    result = adfuller(series.dropna())
    print(f"\nADF Test for {name}")
    print(f"  ADF statistic: {result[0]:.3f}")
    print(f"  p-value:       {result[1]:.4f}")
    print("  → Stationary" if result[1] < 0.05 else "  → Non‑stationary")


In [ ]:
# differencing to make this stationary

def make_stationary(series, name='series'):
    """Run ADF; if non-stationary, difference once and test again."""
    adf_test(series, name)
    if adfuller(series.dropna())[1] < 0.05:
        # Already stationary
        return series, 0
    # 1st difference
    diff1 = series.diff()
    adf_test(diff1.dropna(), name + " (1st diff)")
    if adfuller(diff1.dropna())[1] < 0.05:
        return diff1, 1
    # If still non‑stationary, you could diff again, but usually d<=1 is enough
    return diff1, 1

In [ ]:
# Ensure datetime & sort
meshki_video['timestamp'] = pd.to_datetime(meshki_video['timestamp'])
meshki_video = meshki_video.sort_values('timestamp')

# Daily mean engagement rate
video_daily = (
    meshki_video
    .set_index('timestamp')['videoEngagementRate']
    .resample('D')
    .mean()
    .dropna()
)

video_daily.name = 'video_engagement'

In [ ]:
video_stationary, d_video = make_stationary(video_daily, 'Video Engagement')
print(f"\nUsing d = {d_video} for videos")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_acf(video_stationary.dropna(), lags=30, ax=axes[0])
axes[0].set_title('Video ACF')
plot_pacf(video_stationary.dropna(), lags=30, ax=axes[1])
axes[1].set_title('Video PACF')
plt.tight_layout()
plt.show()

Although the ACF shows slow decay, the ADF test rejects a unit root, indicating the series is stationary but highly persistent; the PACF cutoff at lag 2 suggests modeling this persistence with AR terms rather than differencing.”

In [ ]:
split_idx = int(len(video_daily) * 0.8)
train_video = video_daily.iloc[:split_idx]
test_video  = video_daily.iloc[split_idx:]

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

order_video = (2, d_video, 2)
seasonal_order_video = (0, 0, 0, 0)   # weekly seasonality

model_video = SARIMAX(
    train_video,
    order=order_video,
    seasonal_order=seasonal_order_video,
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_video = model_video.fit(disp=False)
print(results_video.summary())

#Some conclusions:

1. A non‑seasonal ARIMA(2,0,2) captures short‑term dependence in daily video engagement (lags 1 and 2 are significant).

2. Diagnostics show no strong leftover autocorrelation, but residuals are highly heteroskedastic and non‑normal due to rare large spikes.

3. This model is suitable for forecasting baseline / “normal” engagement, but cannot reliably predict viral spikes, which appear as genuine shocks.

In [ ]:
results_video.plot_diagnostics(figsize=(10, 8))
plt.tight_layout()
plt.show()

# Interpretations for graphs above:

1. No strong residual autocorrelation → ARIMA(2,0,2) captures the main temporal structure.

2. Residuals are very non‑normal and heteroskedastic because of one or more huge engagement bursts.

In [ ]:
# Forecast same length as test
n_test = len(test_video)
pred_video = results_video.get_forecast(steps=n_test)
pred_mean_video = pred_video.predicted_mean
pred_ci_video = pred_video.conf_int()

# Align index
pred_mean_video.index = test_video.index
pred_ci_video.index = test_video.index

# Plot
plt.figure(figsize=(12, 5))
plt.plot(train_video.index, train_video, label='Train')
plt.plot(test_video.index, test_video, label='Test', color='black')
plt.plot(pred_mean_video.index, pred_mean_video, label='Forecast', color='red')
plt.fill_between(pred_ci_video.index,
                 pred_ci_video.iloc[:, 0],
                 pred_ci_video.iloc[:, 1],
                 color='red', alpha=0.2)
plt.title('Video Engagement – SARIMA Forecast')
plt.legend()
plt.show()

# Error metrics
mae_v = mean_absolute_error(test_video, pred_mean_video)
rmse_v = np.sqrt(mean_squared_error(test_video, pred_mean_video))
print(f"Video SARIMA – MAE: {mae_v:.2f}, RMSE: {rmse_v:.2f}")


In [ ]:
# Refit on full series for production forecast
final_model_video = SARIMAX(
    video_daily,
    order=order_video,
    seasonal_order=seasonal_order_video,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

steps_ahead = 30
future_forecast_video = final_model_video.get_forecast(steps=steps_ahead)
future_mean_v = future_forecast_video.predicted_mean
future_ci_v = future_forecast_video.conf_int()


## Log transformation

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# --- Prepare daily series ---
meshki_video['timestamp'] = pd.to_datetime(meshki_video['timestamp'])
meshki_video = meshki_video.sort_values('timestamp')

video_daily = (
    meshki_video
    .set_index('timestamp')['videoEngagementRate']
    .resample('D')
    .mean()
    .dropna()
)

# Avoid log(0): add small constant if needed
eps = 1e-6
video_daily_log = np.log(video_daily + eps)
video_daily_log.name = 'video_engagement_log'


In [ ]:
def adf_test(series, name='series'):
    result = adfuller(series.dropna())
    print(f"\nADF for {name}: statistic={result[0]:.3f}, p={result[1]:.4f}")

adf_test(video_daily_log, 'Video log')

# If p > 0.05, difference once
video_log_diff = video_daily_log.diff().dropna()
adf_test(video_log_diff, 'Video log diff')

# Choose d depending on which one is stationary
d_video_log = 1 if adfuller(video_daily_log.dropna())[1] >= 0.05 else 0
series_v = video_daily_log if d_video_log == 0 else video_log_diff
print(f"Using d={d_video_log} for log-video ARIMA")


In [ ]:
split_idx = int(len(series_v) * 0.8)
train_v = series_v.iloc[:split_idx]
test_v  = series_v.iloc[split_idx:]


In [ ]:
order_v = (2, d_video_log, 2)
seasonal_order_v = (0, 0, 0, 0)

model_v = SARIMAX(
    train_v,
    order=order_v,
    seasonal_order=seasonal_order_v,
    enforce_stationarity=False,
    enforce_invertibility=False
)
res_v = model_v.fit(disp=False)
print(res_v.summary())


In [ ]:
fig_diag = res_v.plot_diagnostics(figsize=(16, 10))
fig_diag.suptitle('SARIMAX Log-Video Model — Statsmodels Diagnostics',
                   fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Forecast vs Actual (log scale) ────────────────────────────────────────
forecast_log = res_v.forecast(steps=len(test_v))
forecast_log.index = test_v.index

fig, ax = plt.subplots(figsize=(14, 5))
train_v.plot(ax=ax, label='Train (log)', color='steelblue')
test_v.plot(ax=ax, label='Actual Test (log)', color='darkorange')
forecast_log.plot(ax=ax, label='Forecast (log)', color='crimson', linestyle='--')
ax.set_title('Log-Transformed Video Engagement — Forecast vs Actual')
ax.set_xlabel('Date')
ax.set_ylabel('Log Engagement Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
eps = 1e-6
actual_orig    = np.exp(test_v) - eps
forecast_orig  = np.exp(forecast_log) - eps
train_orig     = np.exp(train_v) - eps

fig, ax = plt.subplots(figsize=(14, 5))
train_orig.plot(ax=ax, label='Train', color='steelblue')
actual_orig.plot(ax=ax, label='Actual Test', color='darkorange')
forecast_orig.plot(ax=ax, label='Forecast', color='crimson', linestyle='--')
ax.set_title('Back-Transformed Video Engagement — Forecast vs Actual')
ax.set_xlabel('Date')
ax.set_ylabel('Engagement Rate')
ax.legend()
plt.tight_layout()
plt.show()